# qBraid-SDK: Transpiler

In [ ]:
import numpy as np

from qbraid.programs import QPROGRAM_REGISTRY
from qbraid.interface import (
    circuits_allclose,
    assert_allclose_up_to_global_phase,
    random_circuit,
)
from qbraid.transpiler import transpile

The qBraid transpiler supports all-to-all connectivity between the following quantum program types:

In [ ]:
QPROGRAM_REGISTRY

## Basic usage example: Qiskit $\leftrightarrow$ Amazon Braket $\leftrightarrow$ Cirq

In [ ]:
from qiskit import QuantumCircuit


def test_circuit():
    circuit = QuantumCircuit(4)

    circuit.h([0, 1, 2, 3])
    circuit.x([0, 1])
    circuit.y(2)
    circuit.z(3)
    circuit.s(0)
    circuit.sdg(1)
    circuit.t(2)
    circuit.tdg(3)
    circuit.rx(np.pi / 4, 0)
    circuit.ry(np.pi / 2, 1)
    circuit.rz(3 * np.pi / 4, 2)
    circuit.p(np.pi / 8, 3)
    circuit.sx(0)
    circuit.sxdg(1)
    circuit.iswap(2, 3)
    circuit.swap([0, 1], [2, 3])
    circuit.cx(0, 1)
    circuit.cp(np.pi / 4, 2, 3)

    return circuit

We'll start with a 4-qubit qiskit circuit that uses 15 unique gates

In [ ]:
qiskit_circuit = test_circuit()
qiskit_circuit.draw()

In [ ]:
braket_circuit = transpile(qiskit_circuit, "braket")
print(braket_circuit)

In [ ]:
cirq_circuit = transpile(qiskit_circuit, "cirq")
print(cirq_circuit)

Qubit indexing varies between packages, so some circuit diagrams appear flipped, but the matrix representations are equivalent.

To verify, we'll use the sdk's `circuits_allclose` function, which applies the agnostic `qbraid.interface.to_unitary` function to each of two input circuits, checks the matricies against `np.allclose`, and returns the result.

In [ ]:
circuits_allclose(qiskit_circuit, braket_circuit) and circuits_allclose(
    braket_circuit, cirq_circuit
)

## Stress-testing against randomly generated circuits

For a second example, we'll generate some even larger circuits, and do so randomly, to test the limits of the transpiler.

The qBraid-SDK has its own `random_circuit` function that takes in any supported package as an argument, but to show that there's no pre-processing or filtering going on behind the scenes, we'll use functions from cirq's testing module to generate circuits and to check equivalance after transpiling.

In [ ]:
import cirq

kwargs = {
    "num_qubits": np.random.randint(8, 11),
    "depth": np.random.randint(8, 11),
    "op_density": np.random.randint(80, 100) / 100,
    "random_state": np.random.randint(1, 11),
}

circuit_start = random_circuit("cirq", **kwargs)
start_u = circuit_start.unitary()
print("num_qubits:", len(circuit_start.all_qubits()))
print("depth:", len(circuit_start))
print("op_density:", kwargs["op_density"])
print(f"matrix dimension: {start_u.shape}\n")
print(circuit_start)

Starting with this randomly generated circuit, we'll repeatedly apply the qbraid circuit wrapper and transpile from one supported package to the next until we arrive all the way back at a cirq circuit.

In [ ]:
braket_circuit = transpile(circuit_start, "braket")
print(type(braket_circuit))

In [ ]:
try:
    pyquil_circuit = transpile(braket_circuit, "pyquil")
    print(type(pyquil_circuit))
except Exception:
    pyquil_circuit = None
    print("PyQuil is not installed. Install via: pip install pyquil")
    print("Skipping pyquil step in the conversion chain.")

In [ ]:
if pyquil_circuit is not None:
    qiskit_circuit = transpile(pyquil_circuit, "qiskit")
else:
    qiskit_circuit = transpile(braket_circuit, "qiskit")
    print("(Bypassed pyquil)")
print(type(qiskit_circuit))

In [ ]:
pytket_circuit = transpile(qiskit_circuit, "pytket")
print(type(pytket_circuit))

In [ ]:
circuit_finish = transpile(pytket_circuit, "cirq")
print(type(circuit_finish))

Computing the final unitary and checking its shape

In [ ]:
finish_u = circuit_finish.unitary()
print(finish_u.shape)

In [ ]:
try:
    assert_allclose_up_to_global_phase(start_u, finish_u, atol=1e-7)
    print("Test passed!")
except AssertionError:
    print("Test failed")